# 1. Import Library
Memasukkan semua library yang dibutuhkan untuk tahap EDA, Preprocessing, Modeling, dan Evaluasi.

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# 2. EDA & Preprocessing

### Load Dataset
Sesuai ketentuan, data **train** hanya digunakan untuk training (termasuk sebagai patokan imputasi dan scaling), dan data **test** khusus digunakan untuk evaluasi akhir. Tidak ada split data lagi di sini.

*(Catatan: Untuk menghindari RAM laptop penuh / Kernel Crash akibat memuat 7,5 juta baris sekaligus, kita langsung mensampling data saat awal load)*

In [32]:
# Mengambil data yang sudah di-split sebelumnya
train_df = pd.read_csv('../dataset/data/train.csv')
test_df = pd.read_csv('../dataset/data/test.csv')

# ================== MEMORY OPTIMIZATION ==================
# Agar laptop tidak crash (Out of Memory), kita ambil 
# 300.000 sampel acak untuk train, dan 50.000 untuk test.
train_limit = min(300000, len(train_df))
test_limit = min(50000, len(test_df))
train_df = train_df.sample(n=train_limit, random_state=42).reset_index(drop=True)
test_df = test_df.sample(n=test_limit, random_state=42).reset_index(drop=True)
# =========================================================

print("Shape of train_df:", train_df.shape)
print("Shape of test_df:", test_df.shape)
train_df.head()

Shape of train_df: (300000, 46)
Shape of test_df: (50000, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-2765989,Source2,1,2018-06-16 13:28:31,2018-06-16 13:58:13,33.813431,-118.167686,NaN,NaN,0.0,...,False,False,False,False,False,False,Day,Day,Day,Day
1,A-3307034,Source2,0,2017-09-14 08:22:22,2017-09-14 09:22:07,39.701771,-86.261589,NaN,NaN,0.0,...,False,False,False,False,False,False,Day,Day,Day,Day
2,A-1212488,Source2,0,2020-12-28 07:08:29,2020-12-28 08:08:36,42.351032,-71.093155,NaN,NaN,0.0,...,False,False,False,False,True,False,Night,Day,Day,Day
3,A-670450,Source2,0,2022-03-21 05:38:42,2022-03-21 06:23:14,30.170641,-95.319542,NaN,NaN,0.0,...,False,False,False,False,True,False,Night,Night,Night,Night
4,A-1798692,Source2,1,2019-11-05 18:56:49,2019-11-05 19:57:59,36.794800,-76.212372,NaN,NaN,0.0,...,False,False,False,False,False,False,Night,Night,Night,Night


### Binarisasi Severity
Mengubah target Severity dari 4 kelas (1,2,3,4) menjadi 2 kelas: 0 (Severity 1-2) dan 1 (Severity 3-4).

In [ ]:
if train_df['Severity'].isin([0, 1]).all():
    print("Severity sudah dalam format biner (0/1), tidak perlu binarisasi ulang.")
else:
    train_df['Severity'] = train_df['Severity'].map({1:0, 2:0, 3:1, 4:1})
    test_df['Severity'] = test_df['Severity'].map({1:0, 2:0, 3:1, 4:1})
    print("Binarisasi Severity selesai (1,2→0; 3,4→1).")

print("Distribusi kelas:")
print("Train:", train_df['Severity'].value_counts().to_dict())
print("Test:", test_df['Severity'].value_counts().to_dict())

### Drop Kolom ID
Menghapus kolom ID karena tidak relevan sebagai fitur.

In [33]:
train_df = train_df.drop(columns=['ID'])
test_df = test_df.drop(columns=['ID'])
print(f"Shape after dropping ID: train {train_df.shape}, test {test_df.shape}")

Shape after dropping ID: train (300000, 45), test (50000, 45)


In [34]:
# Cek missing value SEBELUM imputasi
na_train = train_df.isnull().sum().sum()
na_test = test_df.isnull().sum().sum()
print(f"Missing value sebelum imputasi - train: {na_train}, test: {na_test}")
if na_train > 0 or na_test > 0:
    print("Akan diisi median (numerik) & modus (kategorikal) di tahap 2.1")

Missing value sebelum imputasi - train: 617580, test: 82739
Akan diisi median (numerik) & modus (kategorikal) di tahap 2.1


### 2.1 Missing Value Handling
Penanganan missing value dilakukan dengan mengisi kekosongan (imputasi). Kolom numerik diisi dengan **median**, sedangkan kolom kategorikal diisi dengan **modus**. Ingat, perhitungan median/modus **hanya** didapatkan dari data train.

In [35]:
print("Missing values di data train:\n", train_df.isnull().sum())

# Tentukan mana kolom numerik dan kategorikal
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

# Simpan nilai imputasi untuk deployment
median_impute = {}
for col in numeric_cols:
    median_val = train_df[col].median()
    median_impute[col] = median_val if not pd.isna(median_val) else 0
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)

mode_impute = {}
for col in categorical_cols:
    mode_val = train_df[col].mode()[0]
    mode_impute[col] = mode_val if not train_df[col].mode().empty else 'unknown'
    train_df[col] = train_df[col].fillna(mode_val)
    test_df[col] = test_df[col].fillna(mode_val)

# Verifikasi missing value setelah imputasi
remaining_na_train = train_df.isnull().sum().sum()
remaining_na_test = test_df.isnull().sum().sum()
print(f"Missing value setelah imputasi - train: {remaining_na_train}, test: {remaining_na_test}")
if remaining_na_train == 0 and remaining_na_test == 0:
    print("✅ Tidak ada missing value tersisa. Data siap diproses ke tahap selanjutnya.")
else:
    print("⚠️  Masih ada missing value!")
    print("Train:\n", train_df.columns[train_df.isnull().any()].tolist())
    print("Test:\n", test_df.columns[test_df.isnull().any()].tolist())

# Tampilkan data setelah imputasi
from IPython.display import display
print("\nData train setelah imputasi (sampel):")
display(train_df.head(10))
print(f"Shape: {train_df.shape}")

Missing values di data train:
 Source                        0
Severity                      0
Start_Time                    0
End_Time                      0
Start_Lat                     0
Start_Lng                     0
End_Lat                  169102
End_Lng                  169102
Distance(mi)                  0
Description                   0
Street                      356
City                         11
County                        0
State                         0
Zipcode                     100
Country                       0
Timezone                    274
Airport_Code                891
Weather_Timestamp          4522
Temperature(F)             6385
Wind_Chill(F)             97681
Humidity(%)                6739
Pressure(in)               5493
Visibility(mi)             7188
Wind_Direction             6424
Wind_Speed(mph)           26532
Precipitation(in)        106597
Weather_Condition          7047
Amenity                       0
Bump                          0
Crossing 

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,Source2,1,2018-06-16 13:28:31,2018-06-16 13:58:13,33.813431,-118.167686,37.217042,-87.624424,0.00,Hov lane blocked due to accident on I-405 Sout...,...,False,False,False,False,False,False,Day,Day,Day,Day
1,Source2,0,2017-09-14 08:22:22,2017-09-14 09:22:07,39.701771,-86.261589,37.217042,-87.624424,0.00,Accident on Kentucky Ave at I-74 I-465.,...,False,False,False,False,False,False,Day,Day,Day,Day
2,Source2,0,2020-12-28 07:08:29,2020-12-28 08:08:36,42.351032,-71.093155,37.217042,-87.624424,0.00,Accident on Storrow Dr at Bowker Overpass.,...,False,False,False,False,True,False,Night,Day,Day,Day
3,Source2,0,2022-03-21 05:38:42,2022-03-21 06:23:14,30.170641,-95.319542,37.217042,-87.624424,0.00,Crash on Old Houston Rd at FM-1314.,...,False,False,False,False,True,False,Night,Night,Night,Night
4,Source2,1,2019-11-05 18:56:49,2019-11-05 19:57:59,36.794800,-76.212372,37.217042,-87.624424,0.00,Right lane blocked due to accident on I-64 Ham...,...,False,False,False,False,False,False,Night,Night,Night,Night
5,Source2,0,2017-02-02 20:16:19,2017-02-02 21:01:01,34.219055,-118.195824,37.217042,-87.624424,0.01,Accident on CA-2 Angeles Crest Hwy near Harter...,...,False,False,False,False,False,False,Night,Night,Night,Night
6,Source1,0,2020-01-13 15:10:00,2020-01-13 16:22:42,33.689012,-117.919357,33.689012,-117.919357,0.00,At Harbor Blvd/South Coast Dr/Exit 11 - Accident.,...,False,False,False,False,False,False,Day,Day,Day,Day
7,Source2,1,2018-01-30 06:05:17,2018-01-30 06:34:34,37.578735,-122.048149,37.217042,-87.624424,0.00,Accident on I-880 Southbound before Alvarado B...,...,False,False,False,False,False,False,Night,Night,Night,Day
8,Source2,1,2021-09-09 06:25:34,2021-09-09 06:55:14,41.982349,-87.814209,37.217042,-87.624424,0.00,Lane blocked due to accident on I-90 Kennedy E...,...,False,False,False,False,False,False,Day,Day,Day,Day
9,Source2,1,2021-12-21 14:28:47,2021-12-21 15:14:13,38.359165,-121.972939,37.217042,-87.624424,0.00,Slow lane blocked due to accident on I-80 East...,...,False,False,False,False,False,False,Day,Day,Day,Day


Shape: (300000, 45)


In [36]:
t = train_df
ts = test_df

print("Train:\n", t.isnull().sum())
print("Test:\n", ts.isnull().sum())

Train:
 Source                   0
Severity                 0
Start_Time               0
End_Time                 0
Start_Lat                0
Start_Lng                0
End_Lat                  0
End_Lng                  0
Distance(mi)             0
Description              0
Street                   0
City                     0
County                   0
State                    0
Zipcode                  0
Country                  0
Timezone                 0
Airport_Code             0
Weather_Timestamp        0
Temperature(F)           0
Wind_Chill(F)            0
Humidity(%)              0
Pressure(in)             0
Visibility(mi)           0
Wind_Direction           0
Wind_Speed(mph)          0
Precipitation(in)        0
Weather_Condition        0
Amenity                  0
Bump                     0
Crossing                 0
Give_Way                 0
Junction                 0
No_Exit                  0
Railway                  0
Roundabout               0
Station             

### 2.2 Outliers Handling
Kita menggunakan metode IQR (Interquartile Range) untuk membatasi (capping) nilai outlier agar tidak merusak model. PENTING: Kolom target dilarang keras untuk dicapping!

In [37]:
iqr_bounds = {}
for col in numeric_cols:
    if col == 'Severity':
        continue
        
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    iqr_bounds[col] = {'lower': lower_bound, 'upper': upper_bound}
    
    train_df[col] = np.where(train_df[col] < lower_bound, lower_bound, train_df[col])
    train_df[col] = np.where(train_df[col] > upper_bound, upper_bound, train_df[col])
    
    test_df[col] = np.where(test_df[col] < lower_bound, lower_bound, test_df[col])
    test_df[col] = np.where(test_df[col] > upper_bound, upper_bound, test_df[col])

### 2.3 Encoding
Mengubah data string (teks) menjadi angka numerik yang bisa dipahami model menggunakan `LabelEncoder`. Dibuat secepat mungkin menggunakan dictionary mapping.

In [38]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    
    le_dict = dict(zip(le.classes_, range(len(le.classes_))))
    unknown_val = len(le.classes_)
    label_encoders[col] = {
        'classes': le.classes_.tolist(),
        'mapping': le_dict
    }
    
    le.classes_ = np.append(le.classes_, '<unknown>')
    
    test_df[col] = test_df[col].astype(str).map(le_dict).fillna(unknown_val).astype(int)

### 2.4 Transformasi (Scaling Fitur)
Melakukan standardisasi (rata-rata=0, std=1) agar model yang sensitif pada jarak bekerja lebih optimal.

In [39]:
# TARGET KOLOM DISET KE 'Severity'
TARGET_KOLOM = 'Severity' 

try:
    # Memisahkan Fitur (X) dan Target Label (y)
    X_train_raw = train_df.drop(columns=[TARGET_KOLOM])
    y_train = train_df[TARGET_KOLOM]
    
    X_test_raw = test_df.drop(columns=[TARGET_KOLOM])
    y_test = test_df[TARGET_KOLOM]
    
    # Inisiasi Scaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw) # Scaling test pakai patokan train
except KeyError:
    print("WARNING: Jangan lupa ubah variabel TARGET_KOLOM di atas dengan nama kolom yang benar di dataset ini.")

# 3. Modeling
Membuat model klasifikasi. Kita akan menggunakan algoritma **Random Forest Classifier** karena secara umum algoritma ini tangguh (robust) terhadap struktur data yang belum linear sempurna dan memberikan akurasi yang solid.

In [40]:
try:
    # Menggunakan class_weight='balanced' untuk mengatasi imbalanced data (kelas 1 dan 4 yang langka)
    # Karena di Langkah 2 data sudah di-sample, datanya sudah teracak dengan baik.
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
    
    print(f"Melatih model Random Forest (Balanced) pada seluruh train_df ({len(X_train_scaled)} baris)...")
    rf_model.fit(X_train_scaled, y_train)
    print("Model Random Forest berhasil dilatih!")
except NameError:
    print("Harap lengkapi tahap 2.4 terlebih dahulu.")

Melatih model Random Forest (Balanced) pada seluruh train_df (300000 baris)...
Model Random Forest berhasil dilatih!


# 4. Evaluasi (Test)
Menggunakan fitur dari data uji (test.csv) ke dalam model, lalu membandingkan hasil prediksinya dengan label aslinya untuk mencari nilai Recall, Precision, dan Accuracy.

In [41]:
try:
    print(f"Menguji data test...")
    y_pred = rf_model.predict(X_test_scaled)
    
    # Evaluasi Metriks
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print("=== HASIL EVALUASI MODEL ===")
    print(f"Akurasi   (Accuracy)  : {acc:.4f}")
    print(f"Presisi   (Precision) : {prec:.4f}")
    print(f"Recall    (Recall)    : {rec:.4f}")
    
    # Menampilkan report lebih lengkap (opsional)
    print("\nClassification Report Lengkap:\n")
    print(classification_report(y_test, y_pred, zero_division=0))
except NameError:
    print("Harap lengkapi tahap sebelumnya.")

Menguji data test...
=== HASIL EVALUASI MODEL ===
Akurasi   (Accuracy)  : 0.7409
Presisi   (Precision) : 0.8154
Recall    (Recall)    : 0.7409

Classification Report Lengkap:

              precision    recall  f1-score   support

           0       0.91      0.75      0.82     40173
           1       0.41      0.72      0.52      9827

    accuracy                           0.74     50000
   macro avg       0.66      0.73      0.67     50000
weighted avg       0.82      0.74      0.76     50000



# 5. Simpan Model & Artifacts
Menyimpan model yang sudah dilatih beserta preprocessing artifacts ke folder `model/` untuk digunakan di deployment.

In [42]:
import joblib
import os

model_dir = '../model'
os.makedirs(model_dir, exist_ok=True)

feature_names = X_train_raw.columns.tolist()

preprocessing = {
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'target_column': TARGET_KOLOM,
    'median_impute': median_impute,
    'mode_impute': mode_impute,
    'iqr_bounds': iqr_bounds,
    'label_encoders': label_encoders,
    'feature_names': feature_names
}

joblib.dump(rf_model, os.path.join(model_dir, 'random_forest.pkl'))
joblib.dump(scaler, os.path.join(model_dir, 'scaler.pkl'))
joblib.dump(preprocessing, os.path.join(model_dir, 'preprocessing.pkl'))

print(f"Model dan artifacts berhasil disimpan ke folder '{model_dir}/'")
print("  - random_forest.pkl")
print("  - scaler.pkl")
print("  - preprocessing.pkl")

Model dan artifacts berhasil disimpan ke folder '../model/'
  - random_forest.pkl
  - scaler.pkl
  - preprocessing.pkl
